# RP

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from tqdm import tqdm

In [ ]:
url = "https://www.rp.pl/sitemaps/sitemap.xml"

xml = requests.get(
    url,
    timeout=30
).text

print(xml[:5000])

In [ ]:
xml = requests.get(
    url,
    timeout=30
).text

soup = BeautifulSoup(
    xml,
    "xml"
)

links = [
    x.text.strip()
    for x in soup.find_all("loc")
]

print("sitemaps:", len(links))

for x in links[:100]:
    print(x)

In [ ]:
xml_url = "https://www.rp.pl/sitemaps/sitemap.201807.xml"

xml = requests.get(
    xml_url,
    timeout=30
).text

soup = BeautifulSoup(
    xml,
    "xml"
)

urls = [
    x.text.strip()
    for x in soup.find_all("loc")
]

print("articles:", len(urls))
print()

for x in urls[:20]:
    print(x)

In [ ]:
url = "https://www.rp.pl/sitemaps/sitemap.202202.xml"

xml = requests.get(
    url,
    timeout=30
).text

soup = BeautifulSoup(
    xml,
    "xml"
)

urls = [
    x.text.strip()
    for x in soup.find_all("loc")
]

cats = []

for u in urls:

    path = urlparse(u).path.strip("/")

    if "/" in path:
        cat = path.split("/")[0]
    else:
        cat = "UNKNOWN"

    cats.append(cat)

pd.Series(cats).value_counts().head(100)

In [ ]:
print(
    pd.Series(cats)
    .value_counts()
    .to_string()
)

In [ ]:
xml = requests.get(
    "https://www.rp.pl/sitemaps/sitemap.xml"
).text

soup = BeautifulSoup(
    xml,
    "xml"
)

links = [
    x.text.strip()
    for x in soup.find_all("loc")
]

print(len(links))
print(links[:5])

In [ ]:
targets = {
    "polityka",
    "konflikty-zbrojne",
    "dyplomacja",
    "biznes",
    "spoleczenstwo",
    "akty-prawne",
    "gospodarka",
    "finanse",
    "opinie-polityczno-spoleczne",
    "komentarze",
    "banki",
    "swiat",
    "opinie-ekonomiczne",
    "gielda",
    "rzecz-o-prawie",
    "prawo-karne",
    "dane-gospodarcze",
    "wojsko",
    "opinie-prawne",
    "w-sadzie-i-w-urzedzie",
    "kraj",
    "prawo",
    "historia",
    "podatki",
    "gaz",
    "przestepczosc",
    "praca-emerytury-i-renty",
    "prawo-dla-ciebie",
    "samorzad-i-administracja",
    "dodatki",
    "wypadki",
    "ekonomia",
    "zawody-prawnicze",
    "wegiel",
    "tu-i-teraz",
    "paliwa",
    "nato",
    "surowce-i-paliwa",
    "sluzby",
    "dane-osobowe",
    "ceny-energii",
    "cudzoziemcy",
    "rosja",
    "bezpieczenstwo",
    "energetyka-zawodowa",
    "wydarzenia",
    "analizy-rzeczpospolitej",
    "regulacje-ue",
    "innowacje-w-armiach",
    "energetyka",
    "drony",
    "za-granica"
}

rows = []

for sitemap_url in links:

    if "sitemap." not in sitemap_url:
        continue

    stamp = sitemap_url.split("sitemap.")[-1].split(".xml")[0]

    if not stamp.isdigit():
        continue

    if int(stamp) < 202202:
        continue

    try:

        xml = requests.get(
            sitemap_url,
            timeout=30
        ).text

        soup = BeautifulSoup(
            xml,
            "xml"
        )

        for loc in soup.find_all("loc"):

            url = loc.text.strip()

            part = urlparse(url).path.strip("/").split("/")

            if len(part) == 0:
                continue

            category = part[0]

            if category not in targets:
                continue

            rows.append(
                {
                    "category": category,
                    "url": url
                }
            )

    except Exception as e:

        print(e)

df = pd.DataFrame(rows)

print(df.shape)

print(
    df["category"]
    .value_counts()
    .to_string()
)

df.to_csv(
    "rp_target_categories_urls.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
u = df["url"].iloc[0]

r = requests.get(
    u,
    headers={
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/137.0.0.0 Safari/537.36"
        )
    },
    timeout=30
)

print("=" * 80)
print("URL")
print(u)

print("=" * 80)
print("STATUS")
print(r.status_code)

print("=" * 80)
print("FINAL URL")
print(r.url)

print("=" * 80)
print("CONTENT TYPE")
print(r.headers.get("content-type"))

print("=" * 80)
print("HTML LENGTH")
print(len(r.text))

print("=" * 80)
print("FIRST 2000 CHARS")
print(r.text[:2000])

print("=" * 80)

soup = BeautifulSoup(
    r.text,
    "lxml"
)

imgs = soup.find_all("img")

print("IMG TAGS")
print(len(imgs))

print("=" * 80)

for img in imgs[:20]:

    print(
        img.get("src")
    )

print("=" * 80)

text = soup.get_text(
    " ",
    strip=True
)

print("TEXT LENGTH")
print(len(text))

print("=" * 80)
print(text[:2000])

In [ ]:
sample_df = df.sample(
    min(100, len(df)),
    random_state=42
)

rows = []

for _, row in tqdm(
    sample_df.iterrows(),
    total=len(sample_df)
):

    try:

        r = requests.get(
            row["url"],
            headers={
                "User-Agent": (
                    "Mozilla/5.0 "
                    "(Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 "
                    "(KHTML, like Gecko) "
                    "Chrome/137.0.0.0 Safari/537.36"
                )
            },
            timeout=30
        )

        if r.status_code != 200:
            continue

        soup = BeautifulSoup(
            r.text,
            "lxml"
        )

        text = soup.get_text(
            " ",
            strip=True
        )

        images = []

        for img in soup.find_all("img"):

            for attr in [
                "src",
                "data-src",
                "data-lazy-src",
                "data-original"
            ]:

                value = img.get(attr)

                if not value:
                    continue

                if value.startswith("data:image"):
                    continue

                images.append(value)

        rows.append({
            "category": row["category"],
            "url": row["url"],
            "text_length": len(text),
            "n_images": len(set(images))
        })

    except Exception as e:

        print(e)

check_df = pd.DataFrame(rows)

print(check_df.shape)

print()

print(
    check_df["n_images"]
    .value_counts()
    .sort_index()
)

print()

print(
    "articles with text >1000 chars"
)

print(
    (check_df["text_length"] > 1000)
    .sum()
)

print()

print(
    "articles with images"
)

print(
    (check_df["n_images"] > 0)
    .sum()
)

print()

print(
    "avg images"
)

print(
    round(
        check_df["n_images"].mean(),
        2
    )
)

In [ ]:
sample_df = df.sample(
    min(100, len(df)),
    random_state=42
)

rows = []

for _, row in tqdm(
    sample_df.iterrows(),
    total=len(sample_df)
):

    try:

        r = requests.get(
            row["url"],
            headers={
                "User-Agent":
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137.0 Safari/537.36"
            },
            timeout=30
        )

        if r.status_code != 200:
            continue

        soup = BeautifulSoup(
            r.text,
            "lxml"
        )

        title = None
        publish_date = None
        og_image = None

        m = soup.find(
            "meta",
            property="og:title"
        )

        if m:
            title = m.get("content")

        m = soup.find(
            "meta",
            property="article:published_time"
        )

        if m:
            publish_date = m.get("content")

        m = soup.find(
            "meta",
            property="og:image"
        )

        if m:
            og_image = m.get("content")

        text = soup.get_text(
            " ",
            strip=True
        )

        image_urls = []
        captions = []
        credits = []

        for img in soup.find_all("img"):

            url = (
                img.get("src")
                or img.get("data-src")
                or img.get("data-lazy-src")
                or img.get("data-original")
            )

            if url:

                if not url.startswith("data:image"):

                    image_urls.append(url)

            alt = img.get("alt")

            if alt:

                captions.append(alt)

        for tag in soup.find_all():

            attrs = " ".join(
                tag.attrs.keys()
            ).lower()

            if (
                "author" in attrs
                or "credit" in attrs
                or "copyright" in attrs
            ):

                credits.append(
                    tag.get_text(
                        " ",
                        strip=True
                    )
                )

        rows.append({
            "category": row["category"],
            "url": row["url"],
            "title": title,
            "publish_date": publish_date,
            "text": text,
            "text_length": len(text),
            "og_image": og_image,
            "n_images": len(set(image_urls)),
            "image_urls": list(set(image_urls)),
            "captions": list(set(captions)),
            "credits": list(set(credits))
        })

    except Exception as e:

        print(e)

check_df = pd.DataFrame(rows)

In [ ]:
check_df[
    [
        "title",
        "publish_date",
        "text_length",
        "n_images"
    ]
].head()

In [ ]:
for i in range(5):

    print("=" * 100)

    print(check_df.iloc[i]["title"])

    print()

    print("OG IMAGE")
    print(check_df.iloc[i]["og_image"])

    print()

    print("CAPTIONS")
    print(check_df.iloc[i]["captions"][:5])

    print()

    print("CREDITS")
    print(check_df.iloc[i]["credits"][:5])

    print()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed

rows = []
failed = []


def process_row(row):

    try:

        r = requests.get(
            row["url"],
            headers={
                "User-Agent":
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137.0 Safari/537.36"
            },
            timeout=30
        )

        if r.status_code != 200:

            return (
                False,
                {
                    "category": row["category"],
                    "url": row["url"],
                    "error": f"status_{r.status_code}"
                }
            )

        soup = BeautifulSoup(
            r.text,
            "lxml"
        )

        title = None

        m = soup.find(
            "meta",
            property="og:title"
        )

        if m:
            title = m.get("content")

        text = soup.get_text(
            " ",
            strip=True
        )

        if len(text) < 1000:

            return (
                False,
                {
                    "category": row["category"],
                    "url": row["url"],
                    "error": "short_text"
                }
            )

        return (
            True,
            {
                "category": row["category"],
                "url": row["url"],
                "title": title,
                "text": text,
                "text_length": len(text)
            }
        )

    except Exception as e:

        return (
            False,
            {
                "category": row["category"],
                "url": row["url"],
                "error": str(e)
            }
        )


with ThreadPoolExecutor(
    max_workers=5
) as executor:

    futures = []

    for _, row in df.iterrows():

        futures.append(
            executor.submit(
                process_row,
                row
            )
        )

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        success, result = future.result()

        if success:

            rows.append(result)

        else:

            failed.append(result)


df_texts = pd.DataFrame(rows)

df_failed = pd.DataFrame(failed)

df_texts.to_csv(
    "rp_texts.csv",
    index=False,
    encoding="utf-8-sig"
)

df_failed.to_csv(
    "rp_failed.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df_texts.shape)
print(df_failed.shape)

In [ ]:
df_failed["error"].value_counts().head(20)

In [ ]:
rows_recovered = []

session = requests.Session()

HEADERS = {
    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137.0 Safari/537.36"
}

for _, row in tqdm(
    df_failed.iterrows(),
    total=len(df_failed)
):

    try:

        r = session.get(
            row["url"],
            headers=HEADERS,
            timeout=60
        )

        if r.status_code != 200:
            continue

        soup = BeautifulSoup(
            r.text,
            "lxml"
        )

        title = None

        m = soup.find(
            "meta",
            property="og:title"
        )

        if m:
            title = m.get("content")

        text = soup.get_text(
            " ",
            strip=True
        )

        rows_recovered.append({
            "category": row["category"],
            "url": row["url"],
            "title": title,
            "text": text,
            "text_length": len(text)
        })

    except Exception:
        continue


df_recovered = pd.DataFrame(
    rows_recovered
)

print(df_recovered.shape)

df_recovered.to_csv(
    "rp_recovered.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
df_texts = pd.read_csv(
    "rp_texts.csv"
)

df_recovered = pd.read_csv(
    "rp_recovered.csv"
)

df_all = pd.concat(
    [
        df_texts,
        df_recovered
    ],
    ignore_index=True
)

df_all = df_all.drop_duplicates(
    subset="url"
)

print(df_all.shape)

df_all.to_csv(
    "rp_all_articles.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
print(len(df_texts))
print(len(df_recovered))
print(len(df_all))

In [ ]:
import re

# anchor keywords

anchor_patterns = [

    # ukraine

    r"ukrain\w*",
    r"ukraińc\w*",
    r"ukraińsk\w*",

    # russia

    r"rosj\w*",
    r"rosjan\w*",
    r"rosyjsk\w*",

    # putin

    r"putin\w*",

    # zelensky

    r"zełensk\w*",
    r"zelensk\w*",

    # kremlin

    r"kreml\w*",

    # russian officials

    r"ławrow\w*",
    r"lawrow\w*",
    r"pieskow\w*",
    r"prigożyn\w*",
    r"prigozyn\w*",
    r"miedwiediew\w*",

    # ukrainian officials

    r"kuleba\w*",
    r"budanow\w*"

]

# war / geopolitical context

context_patterns = [

    # war

    r"wojn\w*",
    r"inwazj\w*",
    r"ofensyw\w*",
    r"atak\w*",
    r"szturm\w*",

    # military

    r"żołnierz\w*",
    r"zolnierz\w*",
    r"wojsk\w*",
    r"armi\w*",
    r"broń\w*",
    r"bron\w*",
    r"rakiet\w*",
    r"dron\w*",
    r"czołg\w*",
    r"czolg\w*",
    r"obrona\s+powietrzna\w*",
    r"bombard\w*",

    # nato

    r"nato\w*",

    # occupied territories

    r"donbas\w*",
    r"donieck\w*",
    r"ługańsk\w*",
    r"lugansk\w*",
    r"krym\w*",

    # cities

    r"kijów\w*",
    r"kijow\w*",
    r"charków\w*",
    r"charkow\w*",
    r"odessa\w*",
    r"mariupol\w*",
    r"chersoń\w*",
    r"cherson\w*",
    r"zaporoż\w*",
    r"zaporoz\w*",

    # moscow

    r"moskw\w*",

    # sanctions

    r"sankcj\w*",

    # refugees

    r"uchodźc\w*",
    r"uchodzc\w*",

    # occupation

    r"okupacj\w*",
    r"okupowan\w*",

    # mobilization

    r"mobilizacj\w*",

    # energy

    r"energi\w*",
    r"kryzys\s+energetyczn\w*",
    r"dostaw\w*\s+gazu",
    r"gazociąg\w*",
    r"gazociag\w*",
    r"nord\s*stream\w*"
]

# regex

anchor_regex = re.compile(
    "|".join(anchor_patterns),
    flags=re.IGNORECASE
)

context_regex = re.compile(
    "|".join(context_patterns),
    flags=re.IGNORECASE
)

# filter

mask_anchor = df_all["text"].str.contains(
    anchor_regex,
    na=False
)

mask_context = df_all["text"].str.contains(
    context_regex,
    na=False
)

df_ru_ua = df_all[
    mask_anchor & mask_context
].copy()

print("all articles:", len(df_all))
print("matched:", len(df_ru_ua))

# find matches

def find_matches(text, patterns):

    text = str(text)

    found = []

    for p in patterns:

        if re.search(
            p,
            text,
            flags=re.IGNORECASE
        ):
            found.append(p)

    return found

# save hit words

df_ru_ua["anchor_hits"] = df_ru_ua["text"].apply(
    lambda x: find_matches(
        x,
        anchor_patterns
    )
)

df_ru_ua["context_hits"] = df_ru_ua["text"].apply(
    lambda x: find_matches(
        x,
        context_patterns
    )
)

# save corpus

df_ru_ua.to_csv(
    "rp_russia_ukraine_corpus.csv",
    index=False,
    encoding="utf-8-sig"
)

df_ru_ua.to_csv(
    "rp_russia_ukraine_corpus_with_hits.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df_ru_ua.shape)

df_ru_ua["category"].value_counts().head(30)

In [ ]:
from tqdm import tqdm
from bs4 import BeautifulSoup
import requests

rows = []

session = requests.Session()

for _, row in tqdm(
    df_ru_ua.iterrows(),
    total=len(df_ru_ua)
):

    try:

        r = session.get(
            row["url"],
            headers=HEADERS,
            timeout=60
        )

        if r.status_code != 200:
            continue

        soup = BeautifulSoup(
            r.text,
            "lxml"
        )

        images = set()

        # img tags

        for img in soup.find_all("img"):

            src = img.get("src")

            if src and not src.startswith("data:image"):
                images.add(src)

        # srcset

        for img in soup.find_all("img"):

            srcset = img.get("srcset")

            if srcset:

                for item in srcset.split(","):

                    url = item.strip().split(" ")[0]

                    if url.startswith("http"):
                        images.add(url)

        # preload images

        for link in soup.find_all("link"):

            if link.get("as") == "image":

                srcset = link.get("imagesrcset")

                if srcset:

                    for item in srcset.split(","):

                        url = item.strip().split(" ")[0]

                        if url.startswith("http"):
                            images.add(url)

        rows.append({
            "url": row["url"],
            "n_images": len(images),
            "image_urls": list(images)
        })

    except Exception:
        pass

df_images = pd.DataFrame(rows)

print(df_images.shape)

df_images.to_csv(
    "rp_image_urls.csv",
    index=False,
    encoding="utf-8-sig"
)

# Polityka

In [ ]:
xml = requests.get(
    "https://www.polityka.pl/sitemap.xml",
    timeout=30
).text

soup = BeautifulSoup(xml, "xml")

links = [
    x.text.strip()
    for x in soup.find_all("loc")
]

print("SITEMAPS:", len(links))

for x in links:
    print(x)

In [ ]:
url = "https://www.polityka.pl/sitemap/article0.xml"
print(xml[:3000])

In [ ]:
xml = requests.get(
    "https://www.polityka.pl/sitemap/article0.xml",
    timeout=30
).text

print(xml[:5000])

In [ ]:
xml = requests.get(
    "https://www.polityka.pl/sitemap/article7.xml"
).text

soup = BeautifulSoup(xml, "xml")

urls = [x.text for x in soup.find_all("loc")]

print("count:", len(urls))

for u in urls[-20:]:
    print(u)

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.polityka.pl/tygodnikpolityka/kraj/2346447,1,najlatwiejsze-zadania-nowa-krs-wykonala-teraz-bedzie-juz-trudniej.read"

html = requests.get(url, timeout=30).text
soup = BeautifulSoup(html, "html.parser")

print("og:image tags:")
for m in soup.find_all("meta"):
    if m.get("property") == "og:image":
        print(m.get("content"))

print("\nimg count:", len(soup.find_all("img")))

for img in soup.find_all("img")[:10]:
    print(img.get("src"))

In [ ]:
import requests
from bs4 import BeautifulSoup

for i in [7, 6, 5]:

    xml = requests.get(
        f"https://www.polityka.pl/sitemap/article{i}.xml"
    ).text

    soup = BeautifulSoup(xml, "xml")

    urls = [x.text for x in soup.find_all("loc")]

    print("\nSITEMAP", i)
    print("FIRST:", urls[0])
    print("LAST :", urls[-1])

In [ ]:
from urllib.parse import urlparse
from collections import Counter

cats = Counter()

for url in sample:  # sample = твои 1000 ссылок
    parts = [p for p in urlparse(url).path.split("/") if p]

    # пропускаем мусор
    if len(parts) < 2:
        continue

    # пример:
    # /tygodnikpolityka/kraj/2299999,...
    # /tygodnikpolityka/kultura/...
    # /podkasty/...
    # /opolityce/...

    if parts[0] in {
        "tygodnikpolityka",
        "podkasty",
        "opolityce",
        "pomocnikhistoryczny",
        "niezbednikinteligenta",
    }:
        cat = parts[1]
    else:
        cat = parts[0]

    # отсекаем случаи когда вместо категории попался slug статьи
    if "," in cat or cat.endswith(".read"):
        continue

    cats[cat] += 1

print("\nCATEGORIES\n")

for cat, cnt in cats.most_common():
    print(f"{cnt:5d}  {cat}")

In [ ]:
for u in sample[:20]:
    print(u)

In [ ]:
for u in sample[-20:]:
    print(u)